# 01 — Backtest signal-following vs buy-and-hold

**Disclaimer:** Decision-support only. Not financial advice. See `docs/trading_disclaimer.md`.

This notebook loads 7 days of `prices` and `signals` from TimescaleDB and compares:

1. **Buy & hold** of each symbol over the window.
2. **Signal-following** — go long on BUY, flat on SELL, hold on HOLD.

We do **not** claim the signal-following curve is profitable. In a quiet tape
both strategies tend to track each other; in a volatile tape signal-following
typically underperforms due to fee drag and lag. The value is the **measurement
infrastructure**, not a trade idea.

## What you should see

- Two equity curves overlaid on a Plotly chart.
- A summary table with total return, Sharpe, max DD, and trade count per strategy.
- A clear caveat that this is paper data with last-tick fills.


In [ ]:
import os, sys
from pathlib import Path
PROJECT = Path(os.environ.get("PROJECT_ROOT", "/Users/vishnoiprem/PycharmProjects/data-etl-ml/ai-engineering/realtime-crypto-sentiment"))
sys.path.insert(0, str(PROJECT))
os.chdir(PROJECT)
print("project:", PROJECT)

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from sqlalchemy import text
from src.store import db

# Load 7d of 1m prices for one symbol
SYMBOL = "BTC"
prices = pd.read_sql(
    text("SELECT bucket, close FROM prices_1m WHERE symbol=:s ORDER BY bucket ASC"),
    db.get_engine(),
    params={"s": SYMBOL},
)
prices["bucket"] = pd.to_datetime(prices["bucket"])
print(f"{len(prices):,} minute-bars for {SYMBOL}")
prices.head()

In [ ]:
# Load signals
signals = pd.read_sql(
    text("SELECT ts, action, score, sentiment_zscore, momentum_zscore, reason "
         "FROM signals WHERE symbol=:s ORDER BY ts ASC"),
    db.get_engine(),
    params={"s": SYMBOL},
)
signals["ts"] = pd.to_datetime(signals["ts"])
print(f"{len(signals):,} signals ({signals['action'].value_counts().to_dict()})")
signals.head()

In [ ]:
# Build two equity curves starting from $10k
START = 10_000.0
FEE_BPS = 10

# Align signal timestamps to the minute-bar index
prices = prices.set_index("bucket")

def run_strategy(action_fn):
    eq = START
    position = 0.0
    cost_basis = 0.0
    curve = []
    for ts, row in prices.iterrows():
        px = float(row["close"])
        # Apply any signal at-or-before this minute
        due = signals[signals["ts"] <= ts]
        for _, sig in due.iterrows():
            if sig["action"] == "BUY" and position == 0:
                # go long
                position = eq / px
                cost_basis = px
                eq = 0.0
            elif sig["action"] == "SELL" and position > 0:
                # close
                eq = position * px * (1 - FEE_BPS / 10_000)
                position = 0.0
        # mark-to-market
        m2m = position * px
        curve.append((ts, eq + m2m))
    out = pd.DataFrame(curve, columns=["ts", "equity"]).set_index("ts")
    return out

# 1) Buy & hold: buy at first bar, hold
bh = prices["close"] / prices["close"].iloc[0] * START

# 2) Signal-following
sf = run_strategy(lambda sig: sig["action"])

fig = go.Figure()
fig.add_trace(go.Scatter(x=bh.index, y=bh, name="Buy & hold", line=dict(color="#1f77b4")))
fig.add_trace(go.Scatter(x=sf.index, y=sf["equity"], name="Signal-following", line=dict(color="#2ca02c")))
fig.update_layout(title=f"{SYMBOL} — backtest (paper, last-tick fill, 10 bps fee)",
                  yaxis_title="Equity (USD)", height=450)
fig.show()

In [ ]:
# Summary table
def metrics(equity: pd.Series) -> dict:
    rets = equity.pct_change().dropna()
    if rets.std() == 0 or len(rets) < 2:
        sharpe = 0.0
    else:
        sharpe = float(rets.mean() / rets.std() * np.sqrt(252 * 24 * 60))  # minute-level
    peak = equity.cummax()
    dd = (peak - equity).max()
    return {
        "total_return_%": float((equity.iloc[-1] / equity.iloc[0] - 1) * 100),
        "sharpe": sharpe,
        "max_dd_usd": float(dd),
        "trades": int((signals["action"] != "HOLD").sum()),
    }

summary = pd.DataFrame({
    "buy_hold": metrics(bh),
    "signal_following": metrics(sf["equity"]),
}).T
summary

## Honest interpretation

- **Both curves are noisy.** A 7-day window is too short to be statistically
  meaningful.
- **Sharpe on a 7d, minute-frequency series is unstable.** A single regime shift
  dominates the standard deviation.
- **Fee drag compounds.** 10 bps per fill × N fills is non-trivial over a week
  of small positions.
- **A neutral result is the most likely outcome.** That is fine — the value of
  the lab is iterating from here, not from a single backtest.

## Next steps

- Extend the window to 30 days once the pipeline has been running.
- Add a benchmark against a 50/200 MA crossover.
- Add spread / slippage to the fill model.
- See `02_sentiment_distribution.ipynb` for the input-quality side of the same
  question.
